In [1]:
# Carga de datos
import pandas as pd
import csv
import unicodedata

df_consumo_enero = pd.read_csv('DatosAbiertos_consumohdna_202501.csv', encoding='utf-8', sep=';')
df_consumo_febrero = pd.read_csv('DatosAbiertos_consumohdna_202502.csv', encoding='utf-8', sep=';')
df_consumo_marzo = pd.read_csv('DatosAbiertos_consumohdna_202503.csv', encoding='utf-8', sep=';')

df_consumo = pd.concat([df_consumo_enero,df_consumo_febrero,df_consumo_marzo],
                       ignore_index=True)

In [2]:
# Selección de columnas utiles
columnas_deseadas = ["NRO_DOC_FAC", "PERIODO", "UNIDAD_NEGOCIO", "DEPARTAMENTO", "PROVINCIA", "DISTRITO",
                      "TARIFA", "CARTERA", "IMPORTE", "CONSUMO"]
df_consumo = df_consumo[columnas_deseadas]

In [3]:
# Exploración inicial
display(df_consumo.sample(10))

display(df_consumo.info())

print("\nValores nulos por columna:")
display(df_consumo.isnull().sum())

print("\nDuplicados:", df_consumo.duplicated().sum())

print("\nValores únicos por columna:")
columnas_nominales = ['NRO_DOC_FAC', 'PERIODO', 'UNIDAD_NEGOCIO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO',
                      'TARIFA', 'CARTERA']
limite = 20
for col in columnas_nominales:
    print(f"\nValores únicos en {col}:")
    print(sorted(df_consumo[col].unique())[:limite])

print("\nEstadísticas:")
columnas_numericas = ['IMPORTE', 'CONSUMO']
pd.set_option('display.float_format', lambda x: '{:,.2f}'.format(x))
display(df_consumo[columnas_numericas].describe())

,NRO_DOC_FAC,PERIODO,UNIDAD_NEGOCIO,DEPARTAMENTO,PROVINCIA,DISTRITO,TARIFA,CARTERA,IMPORTE,CONSUMO
98962,S501-68247232,202501,La Libertad,La Libertad,Trujillo,La Esperanza,BT5B,C,6.4,0.0
1959000,S643-07598360,202502,Conchucos,Ancash,Mariscal Luzuriaga,Fidel olivas escudero,BT5B,C,4.6,0.0
1577885,S551-35727200,202502,Chimbote,Ancash,Huarmey,Huarmey,BT5B,C,28.8,39.0
2705552,S630-20472248,202503,Huaraz,Ancash,Carhuaz,Tinco,BT5B,C,8.1,3.0
945666,S643-07569113,202501,Conchucos,Ancash,Carlos F. Fitzcarrald,San luis,BT5B,C,11.5,20.0
2804059,S651-28002280,202503,Cajamarca,Cajamarca,Cajamarca,Los baños del inca,BT5B,C,97.1,100.0
1761117,S651-27709321,202502,Cajamarca,Cajamarca,San Marcos,Jose manuel quiroz,BT5B,C,7.7,3.0
962084,S643-07540090,202501,Conchucos,Ancash,Antonio Raymondi,Chaccho,BT5B,C,30.7,44.0
85031,S501-68111460,202501,La Libertad,La Libertad,Trujillo,La Esperanza,BT5B,C,24.7,30.0
2797017,S651-27951489,202503,Cajamarca,Cajamarca,Cajamarca,Los baños del inca,BT5B,C,54.5,72.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3100476 entries, 0 to 3100475
Data columns (total 10 columns):
 #   Column          Dtype  
---  ------          -----  
 0   NRO_DOC_FAC     object 
 1   PERIODO         int64  
 2   UNIDAD_NEGOCIO  object 
 3   DEPARTAMENTO    object 
 4   PROVINCIA       object 
 5   DISTRITO        object 
 6   TARIFA          object 
 7   CARTERA         object 
 8   IMPORTE         float64
 9   CONSUMO         float64
dtypes: float64(2), int64(1), object(7)
memory usage: 236.5+ MB


None


Valores nulos por columna:


,0
NRO_DOC_FAC,0
PERIODO,0
UNIDAD_NEGOCIO,0
DEPARTAMENTO,0
PROVINCIA,0
DISTRITO,0
TARIFA,0
CARTERA,0
IMPORTE,0
CONSUMO,0



Duplicados: 0

Valores únicos por columna:

Valores únicos en NRO_DOC_FAC:
['S501-68093974', 'S501-68093975', 'S501-68093976', 'S501-68093977', 'S501-68093978', 'S501-68093979', 'S501-68093980', 'S501-68093981', 'S501-68093982', 'S501-68093983', 'S501-68093984', 'S501-68093985', 'S501-68093986', 'S501-68093987', 'S501-68093988', 'S501-68093989', 'S501-68093990', 'S501-68093991', 'S501-68093992', 'S501-68093993']

Valores únicos en PERIODO:
[np.int64(202501), np.int64(202502), np.int64(202503)]

Valores únicos en UNIDAD_NEGOCIO:
['Cajamarca', 'Chimbote', 'Conchucos', 'Huaraz', 'La Libertad', 'La Libertad Norte', 'La Libertad Sierra']

Valores únicos en DEPARTAMENTO:
['Amazonas', 'Ancash', 'Ayacucho', 'Cajamarca', 'Huanuco', 'Junin', 'La Libertad', 'Lambayeque', 'Lima', 'Piura', 'Tumbes']

Valores únicos en PROVINCIA:
['Aija', 'Antonio Raymondi', 'Ascope', 'Bolivar', 'Bolognesi', 'Cajabamba', 'Cajamarca', 'Carhuaz', 'Carlos F. Fitzcarrald', 'Casma', 'Celendin', 'Chachapoyas', 'Chepen', 

,IMPORTE,CONSUMO
count,"3,100,476.00","3,100,476.00"
mean,113.67,131.44
std,"1,302.67","2,387.20"
min,"-10,974.20",0.00
25%,9.80,8.00
50%,41.40,44.00
75%,101.20,107.00
max,"680,676.50","1,390,637.77"


In [4]:
# Normalización de texto (tildes, espacios en blanco y mayúsculas)

import unicodedata

def quitar_tildes(texto):
    if texto is None:
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

columnas_texto = ['NRO_DOC_FAC', 'PERIODO', 'UNIDAD_NEGOCIO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO',
                      'TARIFA', 'CARTERA']


for col in columnas_texto:
    df_consumo[col] = (
        df_consumo[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .apply(quitar_tildes)
    )

In [5]:
# Nueva variable: FECHA
df_consumo["FECHA"] = pd.to_datetime(
    df_consumo["PERIODO"].astype(str),
    format="%Y%m"
)

In [11]:
# Seleccion de columnas finales
columnas_finales = ['FECHA', 'UNIDAD_NEGOCIO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO',
                      'TARIFA', 'CARTERA', 'IMPORTE', 'CONSUMO']

In [13]:
# Exportar datos
df_consumo_limpio = df_consumo[columnas_finales]
df_consumo_limpio.to_parquet('limpio_consumohdna_20251-20253.parquet', index=False)